Assignment:
1. 
2. 
3. 

In [2]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2

sys.path.append("../../")

MARKET_DATA_PATH = "../../market_data/intraday/"

In [3]:
STOCKS = ["AMZN", "GOOG", "MSFT", "GS", "JPM", "MS", "CVX", "XOM"]    # US stocks
TICKERS = ["SPY"] + STOCKS    # all tickers
PERIOD = 5

In [4]:
frames = []
for t in TICKERS:
    fp = f"{MARKET_DATA_PATH}{t}_{PERIOD}mn.csv"
    df = pd.read_csv(fp, index_col=0, parse_dates=True)[["Close"]]
    df.columns = [t]
    frames.append(df)
prices_raw = pd.concat(frames, axis=1).sort_index()

In [5]:
downsample_period = 120

prices = prices_raw.resample(f"{downsample_period}min").last().dropna(how="any")
rets_all = prices.pct_change().dropna(how="any")

# plot correlation matrix (full sample)
fig = px.imshow(rets_all.corr())
fig.show()

In [9]:
cov_matrix = rets_all.cov()*10000*252*6.5*60/120
round(cov_matrix, 1)

,SPY,AMZN,GOOG,MSFT,GS,JPM,MS,CVX,XOM
SPY,205.3,307.3,231.1,211.8,302.3,216.3,291.9,111.7,95.8
AMZN,307.3,919.1,422.5,384.1,404.8,295.2,397.2,138.8,79.4
GOOG,231.1,422.5,767.3,257.5,283.8,190.4,267.4,68.5,33.8
MSFT,211.8,384.1,257.5,519.1,243.8,159.9,231.5,49.8,23.6
GS,302.3,404.8,283.8,243.8,817.1,536.5,718.5,212.7,183.8
JPM,216.3,295.2,190.4,159.9,536.5,556.2,510.3,179.0,154.1
MS,291.9,397.2,267.4,231.5,718.5,510.3,819.4,201.6,174.1
CVX,111.7,138.8,68.5,49.8,212.7,179.0,201.6,401.9,328.9
XOM,95.8,79.4,33.8,23.6,183.8,154.1,174.1,328.9,407.4


In [10]:
# matrix rank
np.linalg.matrix_rank(cov_matrix)


np.int64(9)

In [7]:
# Rolling covariance of US equities (equal-weight universe); returns from pct_change
ROLL_WINDOW = 60  # bars on the resampled grid

rets = rets_all[STOCKS]    # only stocks
n_assets = rets.shape[1]
w = np.ones(n_assets) / n_assets  # equal weights

# Pairwise rolling sample covariance Σ_t for each window end t (MultiIndex: time × asset)
rolling_cov = rets.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).cov()

# Equal-weight basket return; its rolling variance equals w @ Σ_t @ w (same ddof as pandas .var)
basket_rets = rets.mean(axis=1)
rolling_basket_var = basket_rets.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).var(ddof=1)
rolling_index_var = rets_all["SPY"].rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW).var(ddof=1)

last_t = rets.index[-1]
cov_last = rolling_cov.loc[last_t]

# plot rolling variance of equal-weight basket (w′Σw) and SPY index
fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_index_var.index, y=rolling_index_var, mode="lines", name="SPY index"))
fig.add_trace(go.Scatter(x=rolling_basket_var.index, y=rolling_basket_var, mode="lines", name="equal-weight basket"))
fig.update_layout(
    title="Rolling variance of equal-weight basket (w′Σw) and SPY index",
    template="plotly_dark"
)
fig.update_layout(xaxis_title=None, yaxis_title="Basket variance", legend_title="Basket")
fig.show()
